### 05 — Spark SQL Analysis
#### ShieldLife Insurance — Policy Lapse & Retention Analysis
**Purpose:** Perform structured KPI analysis across 6 buckets — 
Portfolio KPIs, Customer, Policy, Channel, Financial, and 
Behavioural — using Spark SQL to answer ShieldLife's core 
business questions on policy lapsation and retention.  
**Input:** Feature-enriched Delta tables from Notebook 04  
**Output:** KPI metrics, segmentation results, and analytical 
insights feeding Tableau dashboards and executive summary  
**Analysis Structure:** KPIs → Customer Bucket → Policy Bucket 
→ Channel Bucket → Financial Bucket → Behavioural Bucket 
→ Product Bucket  
**Platform:** Databricks (Spark SQL)

### Creating Temp View for Analysis

In [0]:
%python
policies = spark.read.format('delta').load('/Volumes/workspace/default/shieldlife_insurance_db/cleaned_data_for_analysis/cleaned_policies_data.delta/', header=True, inferSchema = True)
customers = spark.read.format('delta').load('/Volumes/workspace/default/shieldlife_insurance_db/cleaned_data_for_analysis/cleaned_customers_data.delta/', inferSchema=True, header=True)
fact_events = spark.read.format('delta').load('/Volumes/workspace/default/shieldlife_insurance_db/cleaned_data_for_analysis/cleaned_policy_event_data.delta/', inferSchema=True, header=True)

policies.createOrReplaceTempView('policies')
customers.createOrReplaceTempView('customers')
fact_events.createOrReplaceTempView('events')

In [0]:
select * from policies limit 5;

policy_id,policy_number,product_id,customer_id,channel_id,agent_id,policy_value,policy_tenure,policy_premium,latest_status,issue_date,expiry_date,is_current_policy,cost_spent,total_time_in_grace,issue_year,is_loss_making_policy,is_unreliable_premium,is_future_date,is_invalid_date_range,tenure_bucket,premium_bucket,expected_premium_value,total_premium_received,unrealised_premium_value,is_graced,payment_frequency,has_unrealised_value,is_negative_unrealised,is_renewal_eligible,has_lapsed_date,is_early_defaulted,is_lapsed_after_grace,is_acquisition_at_risk
POL0001571,SL2024001571,3,CUST0001062,1,AGT0730,702000.0,5,116600.0,cancelled,2024-08-02,2026-12-31,true,5792.11,0,2024,false,false,false,false,Short Term (1-5 Yrs),Very High (> ₹50k),583000.0,233200.0,349800.0,false,Quarterly,true,false,false,false,false,false,false
POL0004442,SL2021004442,4,CUST0003018,1,AGT0719,2000.0,1,7900.0,active,2021-09-30,2022-09-28,false,7976.86,0,2021,true,false,false,false,Short Term (1-5 Yrs),Medium (₹5k - ₹19k),7900.0,7900.0,0.0,false,Irregular,false,false,true,false,false,false,false
POL0006870,SL2021006870,2,CUST0004655,1,AGT0218,1268000.0,20,42600.0,lapsed,2021-05-30,2026-12-31,true,5857.85,3,2021,false,false,false,false,Long Term (20-30 Yrs),High (₹20k - ₹50k),852000.0,305630.93,546369.0700000001,true,Monthly,true,false,false,true,false,true,false
POL0012436,SL2023012436,2,CUST0008391,1,AGT0595,1263000.0,15,56700.0,lapsed,2023-01-20,2026-12-31,true,10104.41,27,2023,false,false,false,false,Mid Term (10-15 Yrs),Very High (> ₹50k),850500.0,70875.0,779625.0,true,Quarterly,true,false,false,true,true,true,false
POL0013577,SL2023013577,4,CUST0009157,1,AGT0524,26000.0,3,30000.0,active,2023-10-07,2026-10-07,true,8045.04,0,2023,false,false,false,false,Short Term (1-5 Yrs),High (₹20k - ₹50k),90000.0,85000.0,0.0,false,Monthly,false,false,false,false,false,false,false


In [0]:
select * from customers limit 10

customer_id,customer_name,customer_age,customer_gender,customer_city,customer_occupation,customer_income_bracket,is_invalid_age,age_bucket,customer_latest_status,no_of_policies,expected_lifetime_value,is_renewed_customer
CUST0006132,Yuvraj Sehgal,34.0,Male,Hyderabad,Business Owner,7L-15L,false,Early Career (31–40),Lapsed,1,932000.0,false
CUST0009882,Zansi Balasubramanian,47.0,Female,Hyderabad,Business Owner,<3L,false,Mid Career (41–50),Cancelled,2,1451000.0,false
CUST0010874,Vasana Thakkar,35.0,Male,Bengaluru,Business Owner,7L-15L,false,Early Career (31–40),Active,1,442500.0,true
CUST0015969,Ojas Sagar,44.0,Male,Delhi,Business Owner,15L-30L,false,Mid Career (41–50),Cancelled,3,1359500.0,true
CUST0017183,Kiaan Ramakrishnan,28.0,Male,Bengaluru,Business Owner,15L-30L,false,Young (18–30),Matured,2,2905000.0,false
CUST0044140,Daksh Vyas,46.0,Female,Mumbai,Business Owner,7L-15L,false,Mid Career (41–50),Lapsed,1,1397500.0,false
CUST0060679,Chaman Arya,34.0,Male,Ahmedabad,Business Owner,30L+,false,Early Career (31–40),Lapsed,1,444000.0,false
CUST0070324,Avni Gupta,33.0,Female,Chandigarh,Business Owner,<3L,false,Early Career (31–40),Active,1,699000.0,true
CUST0071615,Chandani Sandhu,51.0,Female,Kolkata,Business Owner,15L-30L,false,Senior (51–65),Matured,1,1144000.0,false
CUST0078276,Manthan Ramakrishnan,33.0,Male,Delhi,Business Owner,7L-15L,false,Early Career (31–40),Claimed,1,1000000.0,false


### KPI Calculation

In [0]:
-- i. Primary: Persistency Ratio = Policies active at end of period / Policies issued at start of period × 100
select 
    round((sum(case when latest_status =='active' then 1 else 0 end ) * 100.0) / count(policy_id), 2) as persistency_ratio
from policies;

persistency_ratio
48.10


In [0]:
-- ii. Customer Renewal Rate 
select round((sum(case when is_renewed_customer then 1 else 0 end) * 100.0) / count(customer_id), 2) as customer_renewal_rate
from customers;

customer_renewal_rate
54.12


In [0]:
-- iii. Policy Renewal Rate
select round(
    try_divide(count(distinct e.policy_id) * 100.0, 
    count(distinct p.policy_id)), 2
) as policy_renewal_rate
from policies p
left join events e 
    on p.policy_id = e.policy_id 
    and e.is_renewed_event = True
where p.expiry_date <= '2026-06-30'
and p.latest_status in ('active', 'matured', 'claimed')

policy_renewal_rate
72.27


### Customer Bucket

In [0]:
-- i. Total Customers
select count(customer_id) from customers;

count(customer_id)
100000


In [0]:
-- ii. Total Active Customers
select 
    sum(case when customer_latest_status = 'Active' then 1 else 0 end) as total_active_customer,
    round((sum(case when customer_latest_status = 'Active' then 1 else 0 end) * 100.0) / count(customer_id), 2) as active_customer_rate
from customers;

total_active_customer,active_customer_rate
48268,48.27


In [0]:
--  iii. Total Lapsed Customer
select 
    sum(case when customer_latest_status = 'Lapsed' then 1 else 0 end) as total_lapsed_customer,
    round((sum(case when customer_latest_status = 'Lapsed' then 1 else 0 end) * 100.0) / count(customer_id), 2) as lapse_rate
from customers;

total_active_customer,active_customer_rate
24811,24.81


In [0]:
-- iv. Total Renewed/Retained Customers (Customers who have renewed there policy)
select 
    sum(case when is_renewed_customer = True then 1 else 0 end) as total_renewed_customers,
    round((sum(case when is_renewed_customer = True then 1 else 0 end) * 100.0) / count(customer_id), 2) as customer_renewal_rate
from customers;

total_active_customer,active_customer_rate
54115,54.12


### Policy Bucket 

In [0]:
-- i. Total Policies Issued
select count(policy_id) from policies;

count(policy_id)
148059


In [0]:
-- ii. Total Policies Lapsed (Policies which has lapsed as latest status)
select 
    sum(case when latest_status = 'lapsed' then 1 else 0 end) as total_lapsed_policies,
    round((sum(case when latest_status = 'lapsed' then 1 else 0 end) * 100.0) / count(policy_id), 2) as lapse_rate
from policies;

total_lapsed_policies,lapse_rate
39801,26.88


In [0]:
-- iii. Total Polices Cancelled
select 
    sum(case when latest_status = 'cancelled' then 1 else 0 end) as total_cancelled_policies,
    round((sum(case when latest_status = 'cancelled' then 1 else 0 end) * 100.0) / count(policy_id), 2) as cancellation_rate
from policies;

total_cancelled_policies,cancellation_rate
14662,9.90


In [0]:
-- iv.	Total Polices Renewed
select count(distinct policy_id) as total_renewed_policies
from events
where is_renewed_event = True

COUNT(DISTINCTpolicy_id)
64017


In [0]:
-- v.	Total Polices Claimed
select 
    sum(case when latest_status = 'claimed' then 1 else 0 end) as total_claimed_policies,
    round((sum(case when latest_status = 'claimed' then 1 else 0 end) * 100.0) / count(policy_id), 2) as claim_rate
from policies;

total_claimed_policies,claim_rate
7457,5.04


In [0]:
-- vi.	Avg Premium Value
select round(avg(policy_premium),2) as avg_premium_value
from policies
where is_unreliable_premium = false;

avg_premium_value
45821.48


In [0]:
-- vii.	Avg Policy Tenure
select round(avg(policy_tenure),2) as avg_policy_tenure
from policies

avg_policy_tenure
14.62


### Channel Bucket

In [0]:
-- i.	Top Policy Selling Channel
select channel_id, count(policy_id) as total_policies_sold from policies group by channel_id; 

channel_id,total_policies_sold
1,83351
3,29698
2,35010


In [0]:
-- ii.	Highest and Lowset Persistency Ratio Channel
select channel_id,
     round((sum(case when latest_status = 'active' then 1 else 0 end)*100.0)/count(policy_id),2) as persistency_ratio
from policies group by channel_id;

channel_id,total_active_polcies
1,45.66
3,53.04
2,49.70


In [0]:
-- Channel distribution for leakage amount and contribution by channel 
select 
    channel_id,
    sum(case when latest_status = 'lapsed' then cost_spent else 0 end) as leakage_amount,
    round(sum(case when latest_status = 'lapsed' then cost_spent else 0 end) * 100.0 / 246437863.08, 2) as per_contribution_to_total_leak
from policies
group by 1
order by leakage_amount desc;


channel_id,leakage_amount,per_contribution_to_total_leak
1,2.0767269237999803E8,84.27
2,3.0977778129999857E7,12.57
3,7787392.569999942,3.16


In [0]:
select 
channel_id, 
sum(case when latest_status = 'lapsed' then cost_spent else 0 end) as total_acquisition_at_risk, 
round((sum(case when latest_status = 'lapsed' then cost_spent else 0 end)*100.0)/(select sum(cost_spent))),2) 
from policies
group by channel_id

channel_id,total_acquisition_at_risk,"round((sum(CASEWHENlatest_status='lapsed'THENcost_spentELSE0END)*100.0)/sum(cost_spent),2)"
1,2.0767269237999803E8,29.3
3,7787392.569999942,21.83
2,3.0977778129999857E7,25.28


### Financial Bucket 

In [0]:
-- i.	Total Premium Received
select sum(total_premium_received) as total_premium_received 
from policies;

total_premium_received
2.997032036548028E10


In [0]:
-- ii.	Total Acquisition Cost Spent & At Risk
select 
    sum(cost_spent) as total_acquisition_cost_spent,
    sum(case when latest_status = 'lapsed' then cost_spent else 0 end) as total_acquisition_at_risk,
    round((sum(case when latest_status = 'lapsed' then cost_spent else 0 end)*100.0)/sum(cost_spent),2) total_acquisition_per_at_risk
from policies; 

total_acquisition_cost_spent,total_acquisition_at_risk,total_acquisition_per_at_risk
8.66880731080104E8,2.4643786307999328E8,28.43


In [0]:
-- iii.	Total Lapsed Amount (Money we received before customer lapsing)
select sum(total_premium_received) as money_received_before_lapse
 from policies where latest_status = 'lapsed'

money_received_before_lapse
7.51457938448998E9


In [0]:
-- iv.	Total Lifetime Value Loss
select sum(unrealised_premium_value) as total_ltv_loss
from policies

total_ltv_loss
2.7216238137730015E10


In [0]:
-- v.	Avg Customer Excpected Lifetime Value
select avg(expected_lifetime_value) as expected_customer_lifetime_value from customers  

avg(expected_lifetime_value)
1004588.929


In [0]:
-- vi.	Lapse Rate by Premium Payment Frequency
select payment_frequency, 
    sum(case when latest_status = 'lapsed' then 1 else 0 end) total_policies_lapsed,
    round((sum(case when latest_status = 'lapsed' then 1 else 0 end) * 100.0) / count(policy_id), 2) as lapsed_rate
from policies 
group by payment_frequency

payment_frequency,total_policies_lapsed,lapsed_rate
Annual,9230,26.90
Quarterly,13604,26.82
Monthly,15917,26.96
Semi Annual,0,0.00
Irregular,1050,26.39


In [0]:
-- vii. Total Excpected Premium 
select sum(expected_premium_value) as total_expected_premium_value from policies

total_expected_premium_value
1.004588929E11


### Behaviour Bucket

In [0]:
-- i. first premium default / early lapse rate (within 12 months)
select 
    sum(case when latest_status = 'lapsed' and is_early_defaulted = true then 1 else 0 end) as total_early_lapsed,
    round(sum(case when latest_status = 'lapsed' and is_early_defaulted = true then 1 else 0 end) * 100.0 / count(policy_id), 2) as early_lapse_rate
from policies;

total_early_lapsed,early_lapse_rate
11691,7.90


In [0]:
-- ii. lapse rate by age
select 
    c.age_bucket,
    count(case when p.latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when p.latest_status = 'lapsed' then 1 end) * 100.0 / count(p.policy_id), 2) as lapse_rate_by_age
from policies p
join customers c on p.customer_id = c.customer_id
group by c.age_bucket;

age_bucket,total_lapsed,lapse_rate_by_age
Unrealistic (<18 or >90),103,25.50
Senior (51–65),2234,26.17
Early Career (31–40),17540,26.91
Elder (65+),17,35.42
Young (18–30),9655,26.92
Mid Career (41–50),10252,26.96


In [0]:
-- iii. lapse rate by profession
select 
    c.customer_occupation,
    count(case when p.latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when p.latest_status = 'lapsed' then 1 end) * 100.0 / count(p.policy_id), 2) as lapse_rate_by_profession
from policies p
join customers c on p.customer_id = c.customer_id
group by c.customer_occupation;

customer_occupation,total_lapsed,lapse_rate_by_profession
Salaried,14436,26.87
Freelancer,2964,26.72
Self-Employed,8247,26.57
Government Employee,4549,27.09
Retired,1892,27.20
Unknown Occupation,1979,26.91
Business Owner,5734,27.17


In [0]:
-- iv. lapse rate after grace period
select 
    count(case when latest_status = 'lapsed' and is_lapsed_after_grace = true then 1 end) as total_lapsed_after_grace,
    round(count(case when latest_status = 'lapsed' and is_lapsed_after_grace = true then 1 end) * 100.0 / count(policy_id), 2) as lapse_rate_after_grace
from policies

total_lapsed_after_grace,lapse_rate_after_grace
39801,26.88


### Product Bucket 

In [0]:
-- i. product with high lapse rate
select 
    product_id,
    count(case when latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when latest_status = 'lapsed' then 1 end) * 100.0 / count(policy_id), 2) as product_lapse_rate
from policies
group by product_id
order by product_lapse_rate desc;

product_id,total_lapsed,product_lapse_rate
2,10931,29.37
3,9400,28.78
1,12877,24.94
4,6593,24.85


In [0]:
-- ii. lapse rate by tenure bucket
select 
    tenure_bucket,
    count(case when latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when latest_status = 'lapsed' then 1 end) * 100.0 / count(policy_id), 2) as lapse_rate_by_tenure
from policies
group by tenure_bucket;

tenure_bucket,total_lapsed,lapse_rate_by_tenure
Long Term (20-30 Yrs),15675,27.11
Short Term (1-5 Yrs),8958,25.82
Mid Term (10-15 Yrs),15168,27.30


In [0]:
-- iii. lapse rate by premium price
select 
    premium_bucket,
    count(case when latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when latest_status = 'lapsed' then 1 end) * 100.0 / count(policy_id), 2) as lapse_rate_by_premium_price
from policies
group by premium_bucket;

premium_bucket,total_lapsed,lapse_rate_by_premium_price
Medium (₹5k - ₹19k),6432,24.55
Short Term (< ₹5k),165,27.87
Very High (> ₹50k),13036,29.03
High (₹20k - ₹50k),20168,26.41


In [0]:
-- iv. lapse rate by product category
select 
    product_id as product_category,
    count(case when latest_status = 'lapsed' then 1 end) as total_lapsed,
    round(count(case when latest_status = 'lapsed' then 1 end) * 100.0 / count(policy_id), 2) as lapse_rate_by_category
from policies
group by product_id;

product_category,total_lapsed,lapse_rate_by_category
3,9400,28.78
1,12877,24.94
2,10931,29.37
4,6593,24.85


In [0]:
select latest_status, round(count(*) * 100.0 / (select count(*) from policies),2)
from policies
group by latest_status;

latest_status,"round(count(*)*100.0/(SELECTcount(*)FROMpolicies),2)"
cancelled,9.90
claimed,5.04
unknown status,0.02
lapsed,26.88
matured,9.92
active,48.10
in_grace,0.14
